# Lekcja 13 - Pamięć Agenta


## Konfiguracja

Ten notatnik pokazuje, jak stworzyć agenta do rezerwacji podróży z **trwałą pamięcią** przy użyciu **Microsoft Agent Framework** (MAF).

Dowiesz się, jak różne typy pamięci agenta — robocza, krótkoterminowa i długoterminowa — wpływają na to, jak agent przechowuje i wykorzystuje informacje podczas rozmów.

**Wymagania wstępne:**
- Projekt Azure AI Foundry z wdrożonym modelem czatu (np. `gpt-4o-mini`).
- Zalogowanie się za pomocą Azure CLI — uruchom `az login` w terminalu.
- `AZURE_AI_PROJECT_ENDPOINT` — punkt końcowy Twojego projektu Azure AI Foundry.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — nazwa wdrożonego modelu.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Rodzaje pamięci agenta

Agenci AI mogą korzystać z różnych rodzajów pamięci, z których każdy pełni inną funkcję:

### Pamięć robocza
Sam wątek rozmowy — wiadomości wymieniane podczas jednej sesji. Agent może odwoływać się do wcześniejszych wiadomości w tym samym wątku, aby zachować spójność. W MAF jest to tworzone za pomocą **`agent.create_session()`**, która zwraca `AgentSession`.

### Pamięć krótkotrwała
Informacje, które utrzymują się przez czas trwania zadania lub sesji, ale nie są przechowywane na stałe. Na przykład, agent może gromadzić fakty podczas wieloetapowej rozmowy planistycznej i używać ich do stworzenia ostatecznego planu podróży.

### Pamięć długotrwała
Preferencje i fakty, które utrzymują się **pomiędzy sesjami**. Powracający użytkownik nie powinien musieć powtarzać swoich ograniczeń dietetycznych czy stylu podróżowania. Pamięć długotrwała jest zazwyczaj wspierana przez zewnętrzne repozytorium — bazę danych, plik lub indeks wektorowy — i udostępniana agentowi za pomocą narzędzi.


## Pamięć robocza z sesjami

Najprostszą formą pamięci jest sesja rozmowy. Gdy przekażesz tę samą sesję (utworzoną za pomocą `agent.create_session()`) do kolejnych wywołań `agent.run()`, agent widzi całą historię tej rozmowy i może przypomnieć sobie wcześniejsze szczegóły.

Stwórzmy agenta podróży i zademonstrujmy pamięć roboczą.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agent poprawnie przypomniał sobie budżet, ponieważ obie wiadomości należą do tej samej sesji. To jest **pamięć robocza** — istnieje tylko przez czas trwania sesji.

### Co się dzieje przy nowym wątku?

Jeśli utworzymy **nową** sesję, agent nie pamięta poprzedniej rozmowy:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Wzorzec Pamięci Długoterminowej

Aby zapamiętać preferencje użytkownika **pomiędzy sesjami**, potrzebujemy trwałego magazynu, który istnieje poza wątkiem rozmowy. Agent uzyskuje dostęp do tego magazynu za pomocą **narzędzi** — funkcji, które może wywoływać, aby zapisywać i pobierać informacje.

Poniżej implementujemy prosty magazyn preferencji w pamięci (w produkcji należałoby go wspierać bazą danych lub indeksem wektorowym) i udostępniamy go jako narzędzia, których agent może używać.

### Architektura
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenariusz 1 — Nowy użytkownik rezerwuje wycieczkę z okazji rocznicy

Sarah odwiedza po raz pierwszy. Agent powinien zapisać jej preferencje za pomocą narzędzi i użyć ich do rekomendacji hoteli.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenariusz 2 — Sarah wraca po kilku tygodniach

Sarah zaczyna **zupełnie nowy wątek** (symulując nową sesję). Pamięć robocza jest pusta, ale długoterminowy magazyn preferencji nadal zawiera jej informacje. Agent powinien je odzyskać i wykorzystać do personalizacji rekomendacji.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Podsumowanie

W tej lekcji nauczyłeś się trzech typów pamięci agenta oraz jak je zaimplementować za pomocą Microsoft Agent Framework:

| Typ pamięci | Mechanizm MAF | Czas życia |
|---|---|---|
| **Robocza** | `agent.create_session()` | Pojedyncza rozmowa |
| **Krótkoterminowa** | Zgromadzony kontekst w wątku | Pojedyncze zadanie / sesja |
| **Długoterminowa** | Zewnętrzny magazyn dostępny przez funkcje `@tool` | Między sesjami |

### Kluczowe wnioski
1. **`agent.create_session()`** zapewnia pamięć roboczą — agent widzi pełną historię rozmowy w ramach sesji.
2. **Nowe sesje tracą kontekst** — bez pamięci długoterminowej agent nie pamięta poprzednich rozmów.
3. **Funkcje `@tool`** wypełniają lukę — pozwalają agentowi zapisywać i pobierać informacje z trwałego magazynu.
4. **Personalizacja poprawia się z czasem** — im więcej preferencji jest przechowywanych, tym lepsze rekomendacje agenta.

### Zastosowania w praktyce
- **Obsługa klienta**: Zapamiętywanie historii i preferencji klienta
- **Asystenci osobowi**: Utrzymywanie kontekstu przez dni lub tygodnie
- **Opieka zdrowotna**: Śledzenie informacji i preferencji pacjenta
- **E-commerce**: Spersonalizowane zakupy na podstawie historii

### Kolejne kroki
- Zamiana słownika w pamięci na bazę danych lub magazyn wektorowy (np. Azure AI Search)
- Dodanie wygasania pamięci dla informacji wrażliwych na czas
- Budowa systemów wieloagentowych z współdzieloną pamięcią
- Eksploracja notatnika Cognee dla pamięci wspieranej przez graf wiedzy


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zastrzeżenie**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Pomimo naszych starań o dokładność, prosimy pamiętać, że tłumaczenia automatyczne mogą zawierać błędy lub nieścisłości. Oryginalny dokument w języku źródłowym należy uznać za źródło wiążące. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z użycia tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
